# Project 1

In addition to answering the bolded questions on Coursera, complete the Python work in this notebook.

This version uses a **local autograder** without an external grading service. Each graded answer is checked by a named Python grading function, and the grading-summary cell reports your current score.

## Python submission
At the end of the notebook, run the **Export Gradescope Submission** cell. It creates a standalone file named `project1_submission.py`, verifies that Python can import it, and (in Google Colab) downloads it automatically.

**Submit `project1_submission.py` to Gradescope.** Do not submit the notebook-converted `.py` file from the normal Jupyter/Colab *Download .py* menu; the final export cell intentionally creates a cleaner file containing only the functions Gradescope needs.

<b>NOTE: Please remember to remove the following line after completing each implementation:</b>

```python
raise NotImplementedError
```

<b>Otherwise, that function will fail the local and Gradescope tests.</b>


## Getting set up
Run the package installation cell only if your environment is missing a dependency. Then run the imports and local-autograder setup cells before beginning the assignment.

Keep `College.csv` in the same working folder as the notebook while completing the assignment. The final `project1_submission.py` file does **not** load `College.csv` when it is imported, which allows Gradescope to test your functions using its own hidden data.


In [ ]:
# Optional dependency installation. Uncomment only if needed.
# %pip install seaborn


The local autograder does not require a Penn ID or an external account. The student name below is optional and is included only for notebook identification.


In [ ]:
STUDENT_NAME = "Student Name"  # Optional


The grading functions are defined immediately after the import cell. Each function records whether its corresponding answer passed and awards one point.


## Imports

It is important for all (or most) imports to go on the top of a notebook so that other users know which packages need to be installed. In projects that use Anaconda, it is also common to see a file named `requirements.txt` listing all the packages that one has to install.

1. First, import all the necessary modules using the import function. For this exercise, we will be mainly using `pandas`. In the future, we will also be using `numpy`, `seaborn`, and `matplotlib`. To learn more about these packages, you can read through the documentation: 

    - https://pandas.pydata.org/
    - https://numpy.org/
    - https://seaborn.pydata.org/
    - https://matplotlib.org/

In [ ]:
# Let's import the relevant Python packages here
# Feel free to import any other packages for this project

#Data Wrangling
import numpy as np
import pandas as pd

#Plotting
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

In [ ]:
from pathlib import Path

# Each local grading check is worth one point. Adjust these values if needed.
_GRADING_POINTS = {
    "college_least_tuition_test": 1,
    "phd_column_test": 1,
    "phd_column_length_test": 1,
    "private_top_10_test": 1,
    "private_column_length_test": 1,
    "penn_test": 1,
    "many_penns_test": 1,
}

_GRADING_RESULTS = {}
_REFERENCE_COLLEGE = None


def _load_reference_college(dataset_path="College.csv"):
    """Load and clean an independent reference copy of the College dataset."""
    global _REFERENCE_COLLEGE

    if _REFERENCE_COLLEGE is not None:
        return _REFERENCE_COLLEGE.copy(deep=True)

    requested = Path(dataset_path)
    candidates = [requested, Path.cwd() / requested.name, Path("/content") / requested.name, Path("/mnt/data") / requested.name]

    # Preserve order while removing duplicate paths.
    unique_candidates = []
    for candidate in candidates:
        resolved = candidate.expanduser()
        if resolved not in unique_candidates:
            unique_candidates.append(resolved)

    source = next((candidate for candidate in unique_candidates if candidate.is_file()), None)
    if source is None:
        searched = ", ".join(str(path) for path in unique_candidates)
        raise FileNotFoundError(
            f"Could not locate {requested.name}. Place it beside the notebook. Searched: {searched}"
        )

    reference = pd.read_csv(source)

    # Support both the assignment's 'Names' column and the common ISLR CSV format.
    if "Names" not in reference.columns:
        first_column = str(reference.columns[0])
        if first_column.startswith("Unnamed") or first_column.lower() in {"college", "name", "names"}:
            reference = reference.rename(columns={reference.columns[0]: "Names"})
        else:
            raise KeyError("The reference dataset must contain a 'Names' column.")

    reference.set_index("Names", inplace=True)
    reference.drop_duplicates(inplace=True)
    reference["Apps"] = reference["Apps"].fillna(0)

    _REFERENCE_COLLEGE = reference.copy(deep=True)
    return reference


def _record_grade(test_name, passed, detail=""):
    """Record and print the result of one grading function."""
    points = _GRADING_POINTS[test_name]
    earned = points if passed else 0
    _GRADING_RESULTS[test_name] = {
        "passed": bool(passed),
        "earned": earned,
        "possible": points,
        "detail": detail,
    }

    status = "PASS" if passed else "FAIL"
    print(f"{status}: {test_name} ({earned}/{points} point)")
    if detail:
        print(f"  {detail}")
    return bool(passed)


def _reference_or_failure(test_name):
    """Return the reference data, recording a readable failure if it cannot load."""
    try:
        return _load_reference_college()
    except Exception as exc:
        _record_grade(test_name, False, f"Reference-data error: {exc}")
        return None


def _grade_dataframe(test_name, answer, expected):
    """Grade a pandas DataFrame with exact row, column, and value ordering."""
    if not isinstance(answer, pd.DataFrame):
        return _record_grade(
            test_name,
            False,
            f"Expected a pandas DataFrame, but received {type(answer).__name__}.",
        )

    try:
        pd.testing.assert_frame_equal(
            answer,
            expected,
            check_dtype=False,
            check_exact=True,
            check_names=True,
        )
    except AssertionError:
        if answer.shape != expected.shape:
            detail = f"Expected shape {expected.shape}, but received {answer.shape}."
        elif list(answer.columns) != list(expected.columns):
            detail = f"Expected columns {list(expected.columns)}, but received {list(answer.columns)}."
        elif list(answer.index) != list(expected.index):
            detail = "The row labels or row order do not match the expected DataFrame."
        else:
            detail = "The DataFrame values or metadata do not match the expected result."
        return _record_grade(test_name, False, detail)

    return _record_grade(test_name, True)


def grade_college_least_tuition(answer):
    """Grade college_least_tuition."""
    test_name = "college_least_tuition_test"
    reference = _reference_or_failure(test_name)
    if reference is None:
        return False

    expected = reference["Outstate"].idxmin() 
    passed = isinstance(answer, str) and answer == expected
    detail = "" if passed else f"Expected {expected!r}, but received {answer!r}."
    return _record_grade(test_name, passed, detail)


def grade_phd_column(answer):
    """Grade phd_column and verify that double brackets produced a DataFrame."""
    test_name = "phd_column_test"
    reference = _reference_or_failure(test_name)
    if reference is None:
        return False
    return _grade_dataframe(test_name, answer, reference[["PhD"]])


def grade_phd_column_length(answer):
    """Grade phd_column_length."""
    test_name = "phd_column_length_test"
    reference = _reference_or_failure(test_name)
    if reference is None:
        return False

    expected = len(reference[["PhD"]])
    passed = isinstance(answer, (int, np.integer)) and int(answer) == expected
    detail = "" if passed else f"Expected {expected}, but received {answer!r}."
    return _record_grade(test_name, passed, detail)


def grade_private_top10(answer):
    """Grade private_top10."""
    test_name = "private_top_10_test"
    reference = _reference_or_failure(test_name)
    if reference is None:
        return False

    expected = reference[["Private", "Top10perc"]].iloc[[15, 16], :]
    return _grade_dataframe(test_name, answer, expected)


def grade_private_column_length(answer):
    """Grade private_column_length."""
    test_name = "private_column_length_test"
    reference = _reference_or_failure(test_name)
    if reference is None:
        return False

    expected = len(reference[["Private", "Top10perc"]].iloc[[15, 16], :][["Private"]])
    passed = isinstance(answer, (int, np.integer)) and int(answer) == expected
    detail = "" if passed else f"Expected {expected}, but received {answer!r}."
    return _record_grade(test_name, passed, detail)


def grade_penn(answer):
    """Grade penn."""
    test_name = "penn_test"
    reference = _reference_or_failure(test_name)
    if reference is None:
        return False

    expected = reference.loc[["University of Pennsylvania"]]
    return _grade_dataframe(test_name, answer, expected)


def grade_many_penns(answer):
    """Grade many_penns."""
    test_name = "many_penns_test"
    reference = _reference_or_failure(test_name)
    if reference is None:
        return False

    expected = reference[reference.index.to_series().str.contains("Penn", regex=False, na=False)]
    return _grade_dataframe(test_name, answer, expected)


def show_grading_summary():
    """Print the current score and the status of every grading function."""
    earned = sum(result["earned"] for result in _GRADING_RESULTS.values())
    possible = sum(_GRADING_POINTS.values())
    completed = len(_GRADING_RESULTS)

    print(f"Score: {earned}/{possible}")
    print(f"Tests run: {completed}/{len(_GRADING_POINTS)}")
    for test_name in _GRADING_POINTS:
        result = _GRADING_RESULTS.get(test_name)
        status = "PASS" if result and result["passed"] else "FAIL" if result else "NOT RUN"
        print(f"- {test_name}: {status}")

    return earned, possible


def reset_grading_results():
    """Clear all recorded local-grader results."""
    _GRADING_RESULTS.clear()
    print("Grading results cleared.")


## Data

In the following exercise, we will familiarize ourselves with Python by studying the College
dataset, which can be found in the file `College.csv`. This dataset contains the following variables
from 777 different universities and colleges in the US:


| Column | Description | 
|:-|:-|
|Private | Public/private indicator|
|Apps | Number of applications received|
|Accept | Number of applicants accepted|
|Enroll | Number of new students enrolled|
|Top10perc | New students from top 10\% of high school class|
|Top25perc | New students from top 25\% of high school class|
|F.Undergrad | Number of full-time undergraduates|
|P.Undergrad | Number of part-time undergraduates|
|Outstate | Out-of-state tuition|
|Room.Board | Room and board costs|
|Books | Estimated book costs|
|Personal | Estimated personal spending|
|PhD | Percent of faculty with Ph.D.’s|
|Terminal | Percent of faculty with terminal degree|
|S.F.Ratio | Student/faculty ratio|
|Perc.alumni | Percent of alumni who donate|
|Expend | Instructional expenditure per student|
|Grad.Rate | Graduation rate|

2. Load the College dataset using `pandas`

In [ ]:
def read_data(input_file):
    return pd.read_csv(input_file)
college = read_data('College.csv')

3. Use the `head()` function to view the data. 

In [ ]:
college.head()

4. Notice that there is a column ‘Names’ of each university’s name. As we don’t want to use these names as predictors, they are natural candidates to index our data. We can do this using the following function:

```Python
#college is the dataframe
college.set_index('Names', inplace = True)```

In [ ]:
def set_index_func(input_file):
    input_file.set_index('Names', inplace = True)
    return input_file
set_index_func(college)

5. Use the `head()` function again. You should now see that the indices have been replaced with the name of each university in the data set. This means that Python has given each row a name corresponding to the appropriate university. Python will not try to perform calculations on the row names.

In [ ]:
college.head()

6. Use the `info()` function to check and produce a numerical summary of your variables.

In [ ]:
college.info()

7. Examine if there are any duplicates and drop them if needed. Hint: It is perfectly fine to observe no duplicates in the dataset, but it is good practice to check anyhow. Before dropping duplicates, it is also good practice to check whether the duplicates contained any different data, since data entry errors frequently occur.

In [ ]:
college.drop_duplicates(inplace=True) 
college.info() #check again

No duplicates were found, as we still have 777 observations per column.

8. Remove instances where there is no value for the ‘Apps’ column, if any, and replace them with 0. This dataframe will henceforth be our original dataframe. Hint: Na means no value, NaN means Not a Number. It is perfectly fine if there are no Na's in our dataset, but we should always check just in case.

In [ ]:
college['Apps'].fillna(0, inplace = True)
college[college['Apps'] == 0]

No na's were found. The original dataframe had no missing values in the first place, but it's always good to check.

9. Find the college with the least out-of-state tuition and name this variable `college_least_tuition`. The variable should return the name of a college, not its tuition. <b>Answer on Coursera: Which college in this dataset has the least amount of out-of-state tuition?</b>

### Important for Gradescope
Gradescope tests the named functions below on additional DataFrames. Write the actual operation inside each function rather than hard-coding the answer observed in `College.csv`.


In [ ]:
# Optional: use this cell to explore the Outstate column before writing the function below.


Use the function below to store your general solution. Gradescope will also test it on unseen data.


In [ ]:
def get_college_least_tuition(df):
    # YOUR CODE HERE
    raise NotImplementedError

print(get_college_least_tuition(college))


In [ ]:
college_least_tuition = get_college_least_tuition(college)
college_least_tuition


In [ ]:
grade_college_least_tuition(college_least_tuition)


10. From the original dataframe, select the ‘PhD’ column and name this variable `phd_column`. Find the length of this column and name this variable `phd_column_length`. <b>Enter the value of phd_column_length on Coursera.</b> Hint: Make sure to use double brackets to select a Pandas dataframe and single brackets to select a Pandas series. You will notice the difference in formatting: Pandas dataframes are neatly formatted, while Pandas series are not formatted. A Pandas dataframe gives extra functionality compared to a Pandas series, such as appending other dataframes and selecting multiple columns. A Pandas series is essentially a Numpy column. 

In [ ]:
# Double brackets should return a DataFrame column.
def convert_to_dataframe_column(input_file):
    # YOUR CODE HERE
    raise NotImplementedError

phd_column = convert_to_dataframe_column(college)


In [ ]:
grade_phd_column(phd_column)


In [ ]:
def retrieve_column_length(input_column):
    # YOUR CODE HERE
    raise NotImplementedError

phd_column_length = retrieve_column_length(phd_column)
print(phd_column_length)


In [ ]:
grade_phd_column_length(phd_column_length)


11. From the original dataframe, select both the ‘Private’ and ‘Top10perc’ columns, and slice them such that only the rows with index 15 and 16 remain. Name this dataframe `private_top10`. From this dataframe, find the length of the filtered ‘Private’ column and name this variable `private_column_length`. <b>Enter the value of private_column_length on Coursera.</b>

In [ ]:
def retrieve_private_top10(input_file):
    # YOUR CODE HERE
    raise NotImplementedError

private_top10 = retrieve_private_top10(college)


In [ ]:
grade_private_top10(private_top10)


In [ ]:
def retrieve_private_column(input_private_top10):
    # YOUR CODE HERE
    raise NotImplementedError

private_column_length = retrieve_private_column(private_top10)


In [ ]:
grade_private_column_length(private_column_length)


12. From the original dataframe, select the row that only contains data about the “University of Pennsylvania”. Note that many other colleges share the same name, but there is only one unique University of Pennsylvania. Name this dataframe `penn`.

In [ ]:
def locate_penn(input_file):
    # YOUR CODE HERE
    raise NotImplementedError

penn = locate_penn(college)


In [ ]:
grade_penn(penn)


13. From the original dataframe, select the rows that contain all colleges with the name “Penn” included. The “P” should be capitalized. Name this dataframe `many_penns`. Comment on your observations. <b>Answer on Coursera: How many universities are there in this dataset that include “Penn”? </b>

In [ ]:
def retrieve_many_penns(input_file):
    # YOUR CODE HERE
    raise NotImplementedError

many_penns = retrieve_many_penns(college)


In [ ]:
grade_many_penns(many_penns)


In [ ]:
show_grading_summary()


*Continue to explore the dataset by using any of the skills you learned in Recitation 1. Keep this notebook in a location that is easily accessible, as we will continue with it for the next assignment.*

## Export Gradescope Submission

After you finish the assignment, run **all cells that define your graded functions**, then run the export cell below.

The export cell creates and downloads **`project1_submission.py`**. **That generated file is the file to upload to Gradescope.**

> Do **not** use Colab's **File → Download → Download .py** for your Gradescope submission. That option converts the entire notebook—including local-grading and notebook-only cells—rather than creating the clean submission module.

The exported file contains only the required assignment functions plus the necessary imports, and it is syntax-checked before download.

In [ ]:
# Export the student's graded functions as a clean, standalone Python module.
import ast
import subprocess
import sys
from pathlib import Path

REQUIRED_SUBMISSION_FUNCTIONS = [
    "read_data",
    "set_index_func",
    "get_college_least_tuition",
    "convert_to_dataframe_column",
    "retrieve_column_length",
    "retrieve_private_top10",
    "retrieve_private_column",
    "locate_penn",
    "retrieve_many_penns",
]


def _latest_function_sources_from_history(function_names):
    """Find the latest notebook definition of each required function."""
    wanted = set(function_names)
    found = {}

    ip = get_ipython()
    history = ip.history_manager.input_hist_raw

    for cell_source in history:
        if not cell_source or not isinstance(cell_source, str):
            continue
        try:
            tree = ast.parse(cell_source)
        except SyntaxError:
            # Ignore notebook-only cells or cells that were incomplete when executed.
            continue

        for node in tree.body:
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name in wanted:
                source = ast.get_source_segment(cell_source, node)
                if source is None:
                    source = ast.unparse(node)
                found[node.name] = source.strip()

    return found


def export_gradescope_submission(filename="project1_submission.py", download=True):
    """Create the clean .py file that students should upload to Gradescope."""
    sources = _latest_function_sources_from_history(REQUIRED_SUBMISSION_FUNCTIONS)
    missing = [name for name in REQUIRED_SUBMISSION_FUNCTIONS if name not in sources]
    if missing:
        raise RuntimeError(
            "Cannot export yet. Run the cells that define these functions, then rerun this cell: "
            + ", ".join(missing)
        )

    header = '''"""Project 1 Gradescope submission.

Generated from the Project 1 notebook. This module intentionally contains no
notebook magics, grading calls, or top-level College.csv loading.
"""

import numpy as np
import pandas as pd

'''

    body = "\n\n\n".join(sources[name] for name in REQUIRED_SUBMISSION_FUNCTIONS)
    submission_source = header + body + "\n"

    # Validate syntax before writing/downloading.
    ast.parse(submission_source, filename=filename)

    output_path = Path(filename).resolve()
    output_path.write_text(submission_source, encoding="utf-8")

    # A second syntax check using the student's Python interpreter.
    subprocess.run(
        [sys.executable, "-m", "py_compile", str(output_path)],
        check=True,
        capture_output=True,
        text=True,
    )

    print(f"Created and verified: {output_path}")
    print("Upload THIS file to Gradescope (not the full notebook converted with File > Download > .py).")

    if download:
        try:
            from google.colab import files
            files.download(str(output_path))
        except ImportError:
            print("Automatic download is only available in Google Colab.")

    return output_path